# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Iqra411/lyrank-ML-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.* Finding A — "What Predicts Health?" (Random Forest feature importance, p.27)

Where does the label come from? Health Score is explicitly a FlyRank composite:
Impressions (30 pts) + Position (30 pts) + CTR (20 pts) + Scroll Depth (20 pts)
(p.5, p.36). The paper itself flags this ("the target itself is partly
constructed from some of these inputs, so importance is descriptive rather
than causal") -- so my question isn't about the leakage, they already named
it. My question is about the SPLIT. Methodology (p.36) states an 80/20 split
for the Random Forest with no mention of grouping by brand. With 57 brands and
61.8K content pieces, an ungrouped 80/20 split almost certainly puts many
pages from the same brand in both train and test -- the same failure mode I
measured directly in Section 2 of this notebook (31/32 clients overlapping,
inflating my own avg_precision from 0.618 to 0.768). Position and Impressions
dominating importance could partly reflect a model that memorized brand-level
baselines rather than a general position-drives-health relationship. Since
Position and Impressions are two of the four inputs that DEFINE Health Score,
this is somewhat expected regardless of split -- but a brand-holdout version
of this same chart would tell us whether the ranking survives leaving a whole
brand's pages out. Constructive ask: report the same importance ranking under
a brand-grouped split alongside the current one, the way the leakage skill
recommends reporting both split numbers side by side.

Finding B — "What Predicts Growth?" (Logistic Regression, 71% holdout accuracy, p.29)

Where does the label come from? Growth/decline comes from trend_direction,
itself derived from 30-day-vs-prev-30-day impression change (p.5) -- the same
label-construction pattern as my own is_declining_label, and the paper
correctly keeps trend_pct-adjacent columns out of the coefficient list, so
no direct label leakage there.

Does the validation design carry the claim? Same 80/20-split question as
Finding A, but the stakes are different here: 71% accuracy is reported as a
single headline number with no base rate stated next to it anywhere on p.29.
The hunting-leakage-and-validating skill is explicit: "Accuracy of 71% on a
label that is 62% positive is 9 points of skill, not 71." My own dataset's
base decline rate is 54.2% (close to a coin flip), but I don't know the
57-brand portfolio's base rate -- if growing vs. declining pages split
unevenly (say 65/35), 71% accuracy could be barely better than always
guessing the majority class. Constructive ask: state the class balance next
to the 71% figure, and given the coefficients lean heavily on
content_age/days_since_update/days_visible (all "1" on a 0-1 scale, everything
else "0" -- a very sparse-looking chart), a brand-grouped holdout would show
whether that separation holds for brands the model never trained on, the same
before/after comparison I ran in Section 2.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*
BEFORE: a naive random 80/20 row split. AFTER: the grouped client-holdout split
from Week 5 (client_id excluded from features, per flyrank-data/SKILL.md).

|              | Random split (BEFORE) | Grouped split (AFTER) |
|--------------|------------------------|------------------------|
| Test rows    | 6,000                  | 2,325                  |
| Test base rate | 0.542                | 0.391                  |
| ROC AUC      | 0.758                  | 0.750                  |
| Avg precision| 0.768                  | 0.618                  |
| Precision@20 | 0.95                   | 0.65                   |
| Precision@50 | 0.90                   | 0.74                   |
| Precision@100| 0.90                   | 0.72                   |

Under the random split, 31 of 32 clients show up in BOTH train and test --
the model can partly memorize a client's typical pattern rather than learn
something general. ROC AUC barely moves (gap 0.008), which on its own would
look reassuring -- but that's exactly the trap: AUC averages over the whole
ranking and hides where the damage is. Precision@20 tells the real story:
0.95 under random-split memorization vs. 0.65 once client leakage is closed,
an 0.30 drop right where a reviewer would actually be looking (top of the
queue). Avg precision drops similarly (0.768 -> 0.618). The honest number is
the grouped one; the random-split number is the inflated one, and I'm
reporting both so the gap itself is on the record.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

RANDOM_STATE = 42
df = pd.read_csv('https://raw.githubusercontent.com/Iqra411/lyrank-ML-internship/main/data/raw/content_refresh_anonymized.csv')

numeric_fill_zero = [
    "search_volume","competition","cpc","word_count","char_count",
    "impressions_90d","clicks_90d","pageviews_90d","sessions_90d","users_90d",
    "engaged_sessions_90d","ai_sessions_90d","scroll_events_90d",
    "days_with_impressions","days_with_sessions","impressions_last_30d",
    "clicks_last_30d","sessions_last_30d","impressions_prev_30d",
    "clicks_prev_30d","sessions_prev_30d","content_age_days","age_tier_order",
    "days_since_last_update","ctr","avg_position","engagement_rate",
    "scroll_rate","ai_traffic_pct","trend_pct",
]
for c in numeric_fill_zero:
    df[c] = pd.to_numeric(df[c], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)

cat_cols = ["competition_level","content_type","main_intent","provider_used","model_used",
            "age_tier","freshness_tier","word_count_tier","char_count_tier",
            "impression_tier","position_tier","trend_direction"]
for c in cat_cols:
    df[c] = df[c].fillna("unknown").astype(str).replace({"": "unknown", "nan": "unknown"})

df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
df = df.drop_duplicates(subset=["content_id"]).reset_index(drop=True)
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"])

MODEL_NUMERIC_FEATURES = [
    "search_volume","competition","cpc","word_count","char_count",
    "log_impressions_90d","log_clicks_90d","log_sessions_90d","log_ai_sessions_90d",
    "days_with_impressions","days_with_sessions","content_age_days",
    "days_since_last_update","ctr","avg_position","engagement_rate",
    "scroll_rate","ai_traffic_pct",
]
MODEL_CATEGORICAL_FEATURES = [
    "competition_level","content_type","main_intent","age_tier",
    "freshness_tier","word_count_tier","impression_tier","position_tier",
]
num_frame = df[MODEL_NUMERIC_FEATURES].apply(pd.to_numeric, errors="coerce").replace([np.inf,-np.inf], np.nan).fillna(0)
cat_frame = pd.get_dummies(df[MODEL_CATEGORICAL_FEATURES].astype(str), prefix=MODEL_CATEGORICAL_FEATURES, dtype=float)
feat = pd.concat([num_frame.reset_index(drop=True), cat_frame.reset_index(drop=True)], axis=1)
target = df["is_declining_label"].astype(int)

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))
    top = np.asarray(y_true)[order[:k]]
    return round(float(top.mean()), 3) if len(top) else 0.0

def run(train_idx, test_idx, label):
    X_train, X_test = feat.iloc[train_idx], feat.iloc[test_idx]
    y_train, y_test = target.iloc[train_idx], target.iloc[test_idx]
    rf = RandomForestClassifier(class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25, n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE)
    rf.fit(X_train, y_train)
    proba = rf.predict_proba(X_test)[:, 1]
    print(f"\n--- {label} ---")
    print("train rows:", len(train_idx), "test rows:", len(test_idx))
    print("test base rate:", round(y_test.mean(), 3))
    print("roc_auc:", round(roc_auc_score(y_test, proba), 3))
    print("avg_precision:", round(average_precision_score(y_test, proba), 3))
    for k in [20,50,100]:
        print(f"precision@{k}:", precision_at_k(y_test, proba, k))
    return roc_auc_score(y_test, proba)

# BEFORE: naive random row split (same client can appear in both train/test)
rand_train_idx, rand_test_idx = train_test_split(np.arange(len(df)), test_size=0.2, random_state=RANDOM_STATE, stratify=target)
auc_random = run(rand_train_idx, rand_test_idx, "BEFORE: random row split (not honest)")

# AFTER: grouped client-holdout split (same as w05)
client_series = df["client_id"].astype(str)
unique_clients = client_series.drop_duplicates().to_numpy()
rng = np.random.default_rng(RANDOM_STATE)
shuffled = rng.permutation(unique_clients)
n_test_clients = max(1, round(len(shuffled) * 0.2))
test_clients = set(shuffled[:n_test_clients])
test_mask = client_series.isin(test_clients).to_numpy()
grp_train_idx = np.where(~test_mask)[0]
grp_test_idx = np.where(test_mask)[0]
auc_grouped = run(grp_train_idx, grp_test_idx, "AFTER: grouped client-holdout split (honest)")

overlap_clients = set(client_series.iloc[rand_train_idx]) & set(client_series.iloc[rand_test_idx])
print(f"\nClients appearing in BOTH train and test under the random split: {len(overlap_clients)} / {len(unique_clients)}")
print(f"AUC gap (random - grouped): {round(auc_random - auc_grouped, 3)}")


--- BEFORE: random row split (not honest) ---
train rows: 24000 test rows: 6000
test base rate: 0.542
roc_auc: 0.758
avg_precision: 0.768
precision@20: 0.95
precision@50: 0.9
precision@100: 0.9

--- AFTER: grouped client-holdout split (honest) ---
train rows: 27675 test rows: 2325
test base rate: 0.391
roc_auc: 0.75
avg_precision: 0.618
precision@20: 0.65
precision@50: 0.74
precision@100: 0.72

Clients appearing in BOTH train and test under the random split: 31 / 32
AUC gap (random - grouped): 0.008


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*
Feature list (scripts/ml_utils.py MODEL_NUMERIC_FEATURES / MODEL_CATEGORICAL_FEATURES)
excludes trend_direction, trend_pct, impressions_last_30d, impressions_prev_30d,
clicks_last_30d, clicks_prev_30d, sessions_last_30d, sessions_prev_30d -- all of
these are either the literal label source or share its 30-day comparison window.
content_id/client_id are used for grouping only, never as features (per
flyrank-data/SKILL.md and the data dictionary's own explicit warning).

Harness sanity check (per hunting-leakage-and-validating/SKILL.md: "deliberately
ADD a leaky feature and watch the score jump"): I injected trend_pct -- the
literal source of the label -- back in as a feature on the honest grouped
split. ROC AUC jumped from 0.750 to a PERFECT 1.000, and it became 82.7% of
total feature importance, dwarfing every real feature. That's the confession
pattern the skill describes almost exactly (collapse from ~1.0 toward ~0.75
when removed) -- confirms my test harness actually catches leakage rather
than silently passing it through.

Attack checklist:
- [x] Timeline: all 26 real features are trailing-90-day aggregates or static
      content properties, knowable at export time -- none is a future window
      relative to the label
- [x] No label-derived columns in features (confirmed by injection test above)
- [x] No product flags / existing-system scores as features (this dataset
      has none; the Week-4 rule itself is a comparison baseline, never an input)
- [x] Population selection: filtered to impressions_90d > 0 and
      content_age_days >= 90 -- this is a design choice (excludes brand-new,
      zero-traffic pages), not outcome-window information, disclosed here
- [x] Split grouped by client_id
- [x] Base rate printed next to every metric (0.391 test base rate, Section 2)
- [x] Top feature importance sanity-checked -- days_with_impressions /
      log_impressions_90d / avg_position leading is plausible, not suspicious
- [x] Metrics recomputed out-of-fold (test clients never seen in training)

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.metrics import roc_auc_score

print("--- Leakage harness sanity check: inject trend_pct (label-derived) ---")
feat_leaky = feat.copy()
feat_leaky["LEAKY_trend_pct"] = df["trend_pct"].values

X_train_leaky = feat_leaky.iloc[grp_train_idx]
X_test_leaky = feat_leaky.iloc[grp_test_idx]
y_train_g = target.iloc[grp_train_idx]
y_test_g = target.iloc[grp_test_idx]

rf_leaky = RandomForestClassifier(class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25, n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE)
rf_leaky.fit(X_train_leaky, y_train_g)
proba_leaky = rf_leaky.predict_proba(X_test_leaky)[:, 1]
print("ROC AUC WITH leaky trend_pct feature:", round(roc_auc_score(y_test_g, proba_leaky), 3))
print("ROC AUC WITHOUT it (honest, from Section 2):", round(auc_grouped, 3))

imp = pd.Series(rf_leaky.feature_importances_, index=feat_leaky.columns).sort_values(ascending=False)
print("Top feature when leaky column present:", imp.index[0], "importance:", round(imp.iloc[0], 3))

--- Leakage harness sanity check: inject trend_pct (label-derived) ---
ROC AUC WITH leaky trend_pct feature: 1.0
ROC AUC WITHOUT it (honest, from Section 2): 0.75
Top feature when leaky column present: LEAKY_trend_pct importance: 0.827


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*
 Rewriting my Week-5 claims to the paper's own standard -- observed / measured
/ directional / decision-support, nothing stronger:

BEFORE (Week 5, too strong): "Random Forest beats the rule cleanly."
AFTER: "On this dataset and split, Random Forest's precision@50 was measured
at 0.74 vs. the Week-4 rule's 0.50 -- a directional improvement, not a
guarantee it generalizes to clients or time periods outside this slice."

BEFORE: "The model catches 74% of declining pages the rule missed."
AFTER: "In this held-out sample, the model recovered 670 of 903 (74.2%)
declining pages the rule's specific threshold missed -- an observed result
on one client split, not a claim about future performance."

BEFORE (implicit in Week-5's framing): "the model predicts decline."
AFTER: "the model estimates decline risk for prioritization -- it is a
decision-support ranking for human review, not a prediction of what Google's
algorithm will do, per DATA_USE.md's own framing rule."

The AUC-barely-moved-but-precision@K-dropped-a-lot result (Section 2) is
itself now stated as a measured gap between two specific splits, not "my
model is honest" -- honesty is a property of the validation design, not
something the model earns once and keeps.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.